# Learning Technique Recommender — Pipeline Walkthrough
This notebook walks through the same pipeline that powers the Streamlit app (`app.py`).
All logic lives in the `src/` package — this notebook just imports, explains, and visualizes.

### Pipeline
1. **Data** — load pre-analyzed grades or compute from your CSV
2. **Syllabus Classifier** — TF-IDF + cosine similarity → course type confidence
3. **Recommender** — rank techniques by historical grade
4. **Integrated System** — `analyze_syllabus(text)` runs the full pipeline
5. **Demo** — example syllabi end-to-end

In [ ]:
import io
import sys
import os

# Ensure we can import from the project root regardless of where Jupyter starts
project_root = os.path.dirname(os.path.abspath("."))
if "src" not in os.listdir("."):
    raise RuntimeError("Run this notebook from the project root (where src/ lives).")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import (
    DEFAULT_TECHNIQUE_GRADES,
    EXAMPLE_SYLLABI,
    LearningStyleSystem,
    LearningTechniqueRecommender,
    SyllabusClassifier,
    compute_technique_grades,
)

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")

## Part 1 — Data
Upload a CSV with columns `Course Type`, `Learning Technique`, `Grade of Module (%)`,
or use the pre-analyzed default dataset.

In [ ]:
def load_data():
    """Upload a CSV in Colab or fall back to the default dataset."""
    if not IN_COLAB:
        print("Running outside Colab — using default pre-analyzed data.")
        return DEFAULT_TECHNIQUE_GRADES

    print("Upload your CSV (Course Type, Learning Technique, Grade of Module (%))")
    try:
        uploaded = files.upload()
        for filename, content in uploaded.items():
            data = pd.read_csv(io.BytesIO(content))
            print(f"Loaded '{filename}' — {data.shape[0]:,} rows")
            return compute_technique_grades(data)
    except Exception as exc:
        print(f"Upload failed ({exc}) — using default data.")
    return DEFAULT_TECHNIQUE_GRADES


def visualize_technique_effectiveness(technique_grades):
    n = len(technique_grades)
    fig, axes = plt.subplots(n, 1, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]
    for ax, (course_type, techs) in zip(axes, technique_grades.items()):
        items = sorted(techs.items(), key=lambda x: x[1], reverse=True)
        names, grades = zip(*items)
        colors = [
            "#2E7D32" if g > 90 else "#1976D2" if g > 80 else "#FFA000" if g > 70 else "#D32F2F"
            for g in grades
        ]
        ax.bar(names, grades, color=colors)
        ax.set_title(course_type, fontweight="bold")
        ax.set_ylabel("Average Grade (%)")
        ax.set_ylim(50, 100)
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        for j, g in enumerate(grades):
            ax.text(j, g + 0.5, f"{g:.1f}%", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()


technique_grades = load_data()
visualize_technique_effectiveness(technique_grades)

## Part 2 — Syllabus Classifier
`SyllabusClassifier` ranks course types by TF-IDF cosine similarity against
hand-crafted keyword descriptions.

In [ ]:
classifier = SyllabusClassifier()
for course_type, score in classifier.classify(EXAMPLE_SYLLABI["Calculus III"]):
    print(f"  {course_type:<48} {score * 100:5.1f}%")

## Part 3 — Recommender
`LearningTechniqueRecommender` ranks techniques by their average historical grade
for a given course type.

In [ ]:
recommender = LearningTechniqueRecommender(technique_grades)

for ct in recommender.available_course_types:
    best = recommender.predict_with_grade(ct)
    print(f"  {ct:<48} → {best['technique']:<28} ({best['expected_grade']:.1f}%)")

## Part 4 — Integrated System
`LearningStyleSystem` combines the classifier and recommender. The combined score is:

```
combined = course_weight × course_match + (1 − course_weight) × (expected_grade / 100)
```

In [ ]:
system = LearningStyleSystem(technique_grades, course_weight=0.5)


def show_results(title, syllabus):
    print(f"\n{'=' * 60}\n  {title}\n{'=' * 60}")
    r = system.analyze_syllabus(syllabus, top_n=5)

    print("\nCourse Type Classification:")
    for i, (ct, score) in enumerate(r["course_type_scores"], 1):
        print(f"  {i}. {ct:<48} {score * 100:5.1f}%")

    print("\nTop Recommended Techniques:")
    for i, t in enumerate(r["top_techniques"], 1):
        print(f"  {i}. {t['technique']:<32} score={t['combined_score']:>5}% "
              f"(grade {t['expected_grade']:.1f}%, match {t['course_match']:.1f}%)")


for title, syl in EXAMPLE_SYLLABI.items():
    show_results(title, syl)

## Part 5 — Run the App
The Streamlit version of this pipeline lives at `app.py` in the project root.
From the project root run:

```bash
pip install -r requirements.txt
streamlit run app.py
```